In [ ]:
import numpy as np
import pandas as pd
import graph_tool.all as gt

from utils.Flow import *


import time
import logging

from utils.HubConnectivityScore import calculate_hcs_pair
from utils.WeightedCheckUtil import *
from utils.SmallWorld import *

from scipy.stats import linregress
import matplotlib.pyplot as plt

import json
import os

In [ ]:
new_input_graphs = [
    'academia_edu',
    'amazon_copurchases/302',
    'anybeat',
    'arxiv_authors/CondMat',
    'arxiv_citation/HepPh',
    'as_skitter',
    'baidu',
    'berkstan_web',
    'caida_as/20071112',
    'chicago_road',
    'citeseer',
    'cora',
    'dblp_cite',
    'dblp_coauthor_snap',
    # 'dbpedia_link', ## Way to big
    'douban',
    'ego_social/gplus_combined',
    'email_enron',
    'email_eu',
    'epinions_trust',
    'flickr_aminer',
    'flickr_growth',
    'flixster',
    'foursquare_friendships/new',
    'gnutella/31',
    'google',
    'google_plus',
    'google_web',
    'hyves',
    'inploid',
    'internet_as',
    'lastfm_aminer',
    'linux',
    'livejournal',
    'livemocha',
    'marker_cafe',
    'marvel_universe',
    'mislove_osn/youtube',
    'myspace_aminer',
    'notre_dame_web',
    'openstreetmap/01-AL-counties-street_networks:01073_Jefferson_County', ## weight needs to be parsed
    'petster',
    'pgp_strong',
    'pokec',
    'python_dependency',
    'roadnet/CA',
    'scotus_majority/2008',
    # 'soc_net_comms', # - seems like repettition
    'social_location/brightkite',
    'stanford_web',
    'trec_web',
    'tree-of-life/9606',
    'twitter',
    'twitter_15m',
    # 'twitter_2009', # - really big network
    'twitter_sample',
    # 'twitter_social', # - too big
    'us_patents',
    # 'wiki_users', # -make positive weights
    # 'wikiconflict', # - make weight positive
    'wikipedia-en-talk',
    'wikipedia_growth',
    'wikipedia_link/az',
    'wikitree',
    'wordnet',
    'yahoo_ads',
    'sp_infectious',
    'genetic_multiplex/Homo',
    ('arxiv_collab/cond-mat-2005', 'value'),
    'prosper',
    'wiki_link_dyn',
    ('mist/ppi_interolog_worm', 'Count_paper'),
    'libimseti',
    'twitter_higgs/reply',
    'dblp_coauthor',
    'twitter_events/NYClimateMarch2014',
    'qa_user/askubuntu_all',
    'mag_history_coauthor/full-proj',
    'mag_geology_coauthor/full-proj',
    'wiki_talk/de',
    'dbpedia_all',
    'dblp_simplices/full-proj',
    ('bitcoin', 'count'),
    'word_adjacency/spanish',
    'facebook_wall',
    'digg_reply',
    'foldoc',
    'fly_hemibrain',
    'lkml_reply',
    'cofe',
    'slashdot_threads',
    'word_assoc',
    'human_brains/BNU1_0025915_2_DTI_DS16784',
    ('us_agencies/aggregate', 'link_counts'),
    'physics_collab/arXiv',
    'topology',
    'us_roads/DE'
]

In [ ]:
# TODO: put in Utils
def is_graph_weighted(graph, weight_key):
    if weight_key not in graph.edge_properties:
        return False
    
    eprop = graph.edge_properties[weight_key]

    unique_values, counts = np.unique(eprop.get_array(), return_counts=True)
    logging.info(f'Graph has {counts} unique values: {unique_values}')

    return len(unique_values) >= 4

In [ ]:
def wmse(y_fit, x, y, w):
    return sum(w * (y_fit - y) ** 2) / sum(w) / len(y_fit)

In [ ]:
# Edit and put in utils
def set_plotting_style():
    plt.rcParams['lines.linewidth'] = 2
    plt.rcParams['lines.markeredgewidth'] = 2
    plt.rcParams['lines.markersize'] = 10
    plt.rcParams['axes.linewidth'] = 2
    plt.rcParams['font.size'] = 20
    plt.rcParams['legend.fontsize'] = 20 * 0.6
    plt.rcParams['figure.subplot.left'] = 0.25
    plt.rcParams['figure.subplot.right'] = 0.95
    plt.rcParams['figure.subplot.bottom'] = 0.2
    plt.rcParams['figure.subplot.top'] = 0.9
    plt.rcParams['xtick.bottom'] = True
    plt.rcParams['xtick.top'] = True
    plt.rcParams['xtick.direction'] = 'in'
    plt.rcParams['xtick.minor.size'] = 4
    plt.rcParams['xtick.minor.width'] = 2
    plt.rcParams['xtick.major.size'] = 8
    plt.rcParams['xtick.major.width'] = 2
    plt.rcParams['ytick.left'] = True
    plt.rcParams['ytick.right'] = True
    plt.rcParams['ytick.direction'] = 'in'
    plt.rcParams['ytick.minor.size'] = 4
    plt.rcParams['ytick.minor.width'] = 2
    plt.rcParams['ytick.major.size'] = 8
    plt.rcParams['ytick.major.width'] = 2

In [ ]:
def calculate_n_plot_power_law(radius, size, folder_path, fig_suffix = ""):
    box_sizes = [rb for rb in radius]

    x = np.array(box_sizes)
    y = np.array(size)
    x_log = np.log10(box_sizes)
    y_log = np.log10(size)

    fit_pl = linregress(x_log, y_log)
    fit_exp = linregress(x, y_log)

    y_log_pl = fit_pl.slope * x_log + fit_pl.intercept
    y_log_exp = fit_exp.slope * x + fit_exp.intercept

    x_plot = np.linspace(x.min(), x.max(), 100)
    x_log_plot = np.log10(x_plot)
    y_log_exp_plot = fit_exp.slope * x_plot + fit_exp.intercept

    set_plotting_style()

    fig = plt.figure(figsize=(6, 6))
    plt.scatter(x_log, y_log)
    plt.plot(x_log, y_log_pl,
             label='$N_\mathrm{B}\propto l_\mathrm{B}^{%.2f}$' % fit_pl.slope)
    plt.plot(x_log_plot, y_log_exp_plot,
             label='$N_\mathrm{B}\propto e^{%.2f l_\mathrm{B}}$' % fit_exp.slope)
    plt.xlabel('$\log(l_\mathrm{B})$')
    plt.ylabel('$\log(N_\mathrm{B})$')
    plt.legend()
    plt.savefig(f"{folder_path}/power_law_fit{fig_suffix}.svg")

    wmse_pl = wmse(y_fit=y_log_pl, x=x_log, y=y_log, w=size)

    wmse_exp = wmse(y_fit=y_log_exp, x=x_log, y=y_log, w=size)

    pl_ex_ratio = wmse_pl / wmse_exp

    return fit_pl.slope, fit_exp.slope, wmse_pl, wmse_exp, pl_ex_ratio

In [ ]:
def get_metrics(graph, graph_name, weight_key, folder_path, is_mst):
    suffix = "mst" if is_mst else "graph"

    json_path = f"{folder_path}/box_{suffix}_result.json"

    f = open(json_path, "r").read()
    json_file = json.loads(f)

    radius = json_file['radius']
    size = json_file['size']

    fit_pl, fit_exp, wmse_pl, wmse_exp, pl_ex_ratio = calculate_n_plot_power_law(
        radius, size, folder_path, f'_{suffix}')

    assortativity_undirected, variance_undirected = calculate_assortativity(
        graph, "total")

    hcs_std, hcs_mean = calculate_hcs_pair(graph)

    swr = SmallWorldResult.build(graph)

    modularity_score = modularity_leiden_alg(graph, weight_key, is_mst)

    return create_series(
        graph_name,
        is_mst,
        modularity_score,
        swr,
        hcs_std,
        hcs_mean,
        assortativity_undirected,
        variance_undirected,
        fit_pl,
        fit_exp,
        wmse_pl,
        wmse_exp,
        pl_ex_ratio
    )

In [ ]:
def load_and_run_metrics(graph_path, graph_name, path, output_file_path, is_mst, weight_key=None):
    graph = gt.load_graph(graph_path)

    if(weight_key is None and is_graph_weighted(graph, 'weight')):
        weight = "weight"
    else:
        weight = weight_key

    metrics_series = get_metrics(graph, graph_name, weight, path, is_mst)

    new_row_df = pd.DataFrame([metrics_series])

    if os.path.isfile(output_file_path):
        old_df = pd.read_pickle(output_file_path)
        combined_df = pd.concat([old_df, new_row_df], ignore_index=True)

    else:
        combined_df = new_row_df

    combined_df.to_pickle(output_file_path)

In [ ]:
def get_graph_and_prepare_metrics(output_file_path, graph_name, weight_key=None):
    try:
        path = prepare_folder(graph_name)

        file_path = get_network_file_path(graph_name, path, False)

        in_separate_process(
            run=load_and_run_metrics,
            withArgs=(file_path, graph_name,  path, output_file_path, False, weight_key),
            log_as=f'GC of {graph_name}'
        )

        mst_path = get_network_file_path(graph_name, path, True)
        
        if os.path.isfile(mst_path):
            in_separate_process(
                run=load_and_run_metrics,
                withArgs=(mst_path, graph_name, path, output_file_path, True, weight_key),
                log_as=f'MST of {graph_name}'
            )

    except Exception as ex:
        logging.error(f'failed to run: {graph_name} with {ex}')

In [ ]:
def start_pipeline(data):
    config_logging(into='metrics_collection.log')

    logging.info('Starting metrics pipeline')
    start_time = time.time()

    output_df_path = os.path.join(analysis_folder, 'feb26_net_results.pk1')

    for graph_data in data:
        if (isinstance(graph_data, tuple) and len(graph_data) >= 2):
            in_separate_process(
                run=get_graph_and_prepare_metrics,
                withArgs=(output_df_path, graph_data[0], graph_data[1],),
                log_as=graph_data[0]
            )
        else:
            in_separate_process(
                run=get_graph_and_prepare_metrics,
                withArgs=(output_df_path, graph_data,),
                log_as=graph_data
            )

    end_time = time.time()

    compute_time = end_time - start_time

    readable_time = time.strftime("%H:%M:%S", time.gmtime(compute_time))

    logging.info(
        f'Metrics collection finished at {end_time} and took: {readable_time}')

    return compute_time

In [ ]:
start_pipeline(
    new_input_graphs
)